In [1]:
import sys
import pickle
import numpy as np
import torch
from transformers import DistilBertTokenizer, DistilBertModel
from tqdm import tqdm

DATA_PATH = '/Users/jl/MISA/datasets/MOSEI'

# Load tokenizer and model
print("Loading DistilBERT...")
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')
bert_model.eval()
print("Loaded.")

def extract_bert_features(actual_words, max_len=64):
    """Extract CLS token embedding from DistilBERT for a list of words."""
    text = ' '.join(actual_words)
    tokens = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        output = bert_model(**tokens)
    # Use CLS token (first token) as sentence representation
    cls_embedding = output.last_hidden_state[:, 0, :].squeeze().numpy()
    return cls_embedding

# Process each split
for split in ['train', 'dev', 'test']:
    print(f"\nProcessing {split}...")
    with open(f'{DATA_PATH}/{split}.pkl', 'rb') as f:
        data = pickle.load(f)
    
    enriched = []
    for i, ((words, vis, acou, actual_words), label, vid) in enumerate(tqdm(data)):
        bert_feat = extract_bert_features(actual_words)
        enriched.append(((words, vis, acou, actual_words, bert_feat), label, vid))
    
    with open(f'{DATA_PATH}/{split}_bert.pkl', 'wb') as f:
        pickle.dump(enriched, f)
    print(f"Saved {split}_bert.pkl — {len(enriched)} samples")

# Free memory
del bert_model
torch.cuda.empty_cache()
print("\nDone! DistilBERT features extracted and saved.")

Loading DistilBERT...


/Users/jl/micromamba/envs/multimodal/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

...

Saved train_bert.pkl — 300 samples

Processing dev...


100%|██████████| 50/50 [00:01<00:00, 44.71it/s]


Saved dev_bert.pkl — 50 samples

Processing test...


100%|██████████| 100/100 [00:01<00:00, 50.85it/s]

Saved test_bert.pkl — 100 samples

Done! DistilBERT features extracted and saved.
